In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table

In [3]:
cols = [
    "model",
    "use_instructions",
    "type_of_demonstrations",
    "number_of_demonstrations",
    "theme_accuracy",
    "topic_accuracy",
    "concept_accuracy",
    "total_accuracy",
    "theme_precision",
    "topic_precision",
    "concept_precision",
    "theme_recall",
    "topic_recall",
    "concept_recall",
    "theme_f1",
    "topic_f1",
    "concept_f1",
]

def score_table(jobs, cols, measure_hallucination_detection=True):
    table = {col_name: [] for col_name in cols}
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append accuracy scores
        for name, score in calculate_accuracy(df).items():
            table[name].append(score)

        # Append precision, recall, f1 per label
        metrics = calculate_metrics(df)
        for name, score in calculate_metrics(df, measure_hallucination_detection=measure_hallucination_detection).items():
            table[name].append(score)

    return table

In [4]:
def aggregate_results(df, by_cols, cols, funs=None):
    grouped = df.groupby(
        by=by_cols,
        as_index=True,
    )
    
    if funs is None:
        funs = [
        "mean",
        "std",
        "count"
    ]

    agg = grouped.agg({col: funs for col in cols})

    return agg

In [5]:
def bold_extreme_values(s):
    # Bold max for mean
    if "mean" in s.name:
        is_max = s == s.max()
        return ['font-weight: bold' if v else '' for v in is_max]
    if "std" in s.name:
        is_min = s == s.min()
        return ['font-weight: bold' if v else '' for v in is_min]

    return ['' for v in s]

In [6]:
# Collect raw data
basepath = "./outputs/v1"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten
print(len(jobs_list))

res = pd.DataFrame(score_table(jobs_list, cols))
res = prettify_table(res)

220


In [14]:
# Show all columns
pd.options.display.max_columns = None
# Set float formatting
pd.options.display.float_format = '{:,.3f}'.format

agg = aggregate_results(res, ["model", "number_of_demonstrations", "type_of_demonstrations", "use_instructions"], cols[4:])
# Drop count
agg = agg.loc[['Qwen2.5-72B-Instruct'], (slice(None), ['mean'])]
# Highlight max values for each column
agg.style.apply(bold_extreme_values, axis=0)

In [8]:
# Collect raw data
basepath = "./outputs/"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten
print(len(jobs_list))

res2 = pd.DataFrame(score_table(jobs_list, cols))
res2 = prettify_table(res2)

11


In [9]:
# Show all columns
pd.options.display.max_columns = None
# Set float formatting
pd.options.display.float_format = '{:,.3f}'.format

agg = aggregate_results(res2, ["model", "number_of_demonstrations", "type_of_demonstrations", "use_instructions"], cols[4:])
# Drop count
agg = agg.loc['Qwen2.5-7B-Instruct':, (slice(None), ['mean'])]
# Highlight max values for each column
agg.style.apply(bold_extreme_values, axis=0)